# 十二、torch.compile 深度解析

上一章用 profiler 学会了"先测量，再优化"。当你确认瓶颈是**算子太碎、kernel 启动开销大、显存反复读写**时，`torch.compile` 是最省事的武器——一行代码，通常就能带来可观的加速：

```python
model = torch.compile(model)   # 就这一行
```

但"一行代码"背后是一整套编译栈。本章把它拆开讲透：为什么能加速、编译栈怎么运作、什么时候会失效（图断裂）、怎么调试、三种 mode 怎么选、以及和生产部署用的 `torch.export` 有什么区别。

#### 学习目标
1. 理解 eager 模式的性能痛点与编译加速的本质（算子融合）；
2. 掌握编译栈三层：TorchDynamo（图捕获）→ AOTAutograd（前反向图）→ Inductor（代码生成）；
3. 理解图断裂（graph break）、guard、重编译这三个核心概念及其影响；
4. 会用 `TORCH_LOGS`、`torch._dynamo.explain` 等工具诊断编译问题；
5. 掌握三种 mode 的取舍与动态 shape 的处理；
6. 能正确 benchmark 编译后的模型（warmup、避免踩坑）。

#### 一、为什么需要编译：eager 模式的性能痛点

PyTorch 默认以 **eager 模式（即时执行）** 运行：你写 `y = gelu(x @ w + b)`，PyTorch 就立刻依次启动——矩阵乘、加法、激活——**三个独立的 GPU kernel**。每个 kernel 都有固定启动开销，且中间结果要写回显存再读出，大量时间浪费在"调度"和"搬运"上，而不是真正的计算。

这就像做一道菜，每放一种调料都重新开一次火、洗一次锅——开销全在准备动作上。

**编译的核心思想**：把一段连续的算子序列**先捕获成一张计算图**，交给编译器做优化——把能合并的算子融合成一个 kernel（算子融合，operator fusion），再生成针对硬件优化的代码。数据是"算完一次、留在寄存器/缓存里接着算"，不再反复进出显存。

```
eager:   x @ w  →写显存→  读显存 + b  →写显存→  读显存 gelu  → 结果
                  (kernel1)          (kernel2)          (kernel3)

compile: x @ w → +b → gelu  融合为一个 Triton kernel，一次读、一次写
```

> 类比（承接上一章的烹饪比喻）：eager 是"每道菜单独开火"，compile 是"把能一起下锅的拼成一锅"。

#### 二、编译栈全景：三层流水线

`torch.compile` 不是一个单一的优化器，而是一条分层流水线。理解每层干什么，是理解它行为的关键：

| 层次 | 名字 | 做什么 |
|------|------|--------|
| 第 1 层 | **TorchDynamo** | 捕获层。在 Python 字节码层面拦截执行，把能编译的张量运算捕获成 **FX 图**；捕获不了的地方就"图断裂"，回退到 eager |
| 第 2 层 | **AOTAutograd** | 求导层。把前向图展开成**前向 + 反向的联合图**，让编译器能同时优化两个方向、做激活重算规划 |
| 第 3 层 | **Inductor** | 后端层（默认）。把图 lower 成自己的 IR，做算子分解与融合，最终生成 **Triton（GPU）/ C++ OpenMP（CPU）** 代码 |

```
Python 代码
   │  TorchDynamo：字节码捕获 + guard
   ▼
FX Graph（可编译的子图，断裂处回退 eager）
   │  AOTAutograd：构建前反向联合图、功能化
   ▼
前向图 + 反向图
   │  Inductor：分解 / 融合 / 调度 / 代码生成
   ▼
Triton kernel (GPU) / C++ kernel (CPU)  →  执行
```

**三个要记住的点**：
- Dynamo 的工作依赖**运行时观察**：它按"这一次看到的输入条件"录一张图，并附带一堆 **guard**（守卫条件）——下次调用条件相同就复用，不同就重编译；
- AOTAutograd 让反向也能被编译，所以训练也能加速（不只是推理）；
- Inductor 是默认后端，但可以替换（见后文自定义后端）。

#### 三、快速上手与正确 benchmark

用法极简：把模型或函数包一层即可。但**测量加速效果时有讲究**——第一次调用会触发编译（可能几十秒），必须 warmup 之后再计时，否则测的是编译时间不是运行时间。

In [ ]:
import torch
import torch.nn as nn
import time

# 一个便于观察的小模型
model = nn.Sequential(
    nn.Linear(512, 512), nn.ReLU(),
    nn.Linear(512, 512), nn.ReLU(),
    nn.Linear(512, 10),
)
x = torch.randn(256, 512)

# eager 基准
def bench(fn, x, n=20, warmup=5):
    for _ in range(warmup):          # 预热：消除首次开销
        fn(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        fn(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000

eager_ms = bench(model, x)
print(f"eager: {eager_ms:.2f} ms/step（CPU 上 compile 加速通常不明显，GPU 上更显著）")

try:
    compiled = torch.compile(model)
    compile_ms = bench(compiled, x)   # bench 内含 warmup，已把编译时间排除
    print(f"compiled: {compile_ms:.2f} ms/step")
except Exception as e:
    print("torch.compile 不可用:", type(e).__name__, e)

> **benchmark 三原则**：① 先 warmup 再计时（排除编译开销）；② GPU 上计时前后要 `torch.cuda.synchronize()`（否则测的是异步入队时间）；③ 用相同输入多跑几次取平均。这三点错了，测出来的"加速"毫无意义。

#### 四、图断裂（graph break）：加速的头号杀手

TorchDynamo 会尽量把函数里所有张量运算捕获成**一张** FX 图，但遇到无法静态捕获的 Python 操作时会**断开**——断之前的部分编译成一张图，断开的操作回退到 eager，之后再重新捕获下一张图。这就是**图断裂**。

**最常见的断裂原因**：
- 依赖张量**具体数值**的 `if` 分支（如 `if x.sum() > 0`）——编译期不知道条件结果；
- 函数里的 `print` / 日志；
- `.item()` 把张量取成 Python 标量；
- 调用 PyTorch 看不到的 C 扩展、外部库操作。

图断裂为什么伤性能？每断一次就在编译图和 eager 之间来回切换，还丢失了跨断点的算子融合机会。断点如果落在 transformer block 内部（交换大中间张量的地方），代价尤其大。

#### 如何发现图断裂

用 `torch._dynamo.explain()` 直接列出断点数量和原因：

In [ ]:
import torch
import torch._dynamo as dynamo

def toy(a, b):
    x = a / (torch.abs(a) + 1)
    print("hello")          # 断点1：print
    if b.sum() < 0:         # 断点2：依赖张量数值的分支
        b = b * -1
    return x * b

explanation = dynamo.explain(toy)(torch.randn(4), torch.randn(4))
print("图数量:", explanation.graph_count)
print("图断裂数:", explanation.graph_break_count)
print("断点原因:")
for r in explanation.break_reasons:
    print("  -", str(r.reason)[:80])

#### 如何减少图断裂

| 手段 | 做法 |
|------|------|
| 改写条件 | 把依赖数值的 `if` 改成无分支的张量运算，如 `torch.where(cond, a, b)` |
| 移出函数 | 把 `print`、日志、`item()` 挪到被编译函数**外面** |
| `fullgraph=True` | 强制整函数捕获，有断点就**直接报错**——适合用报错来定位并消除断点 |
| `torch.compiler.disable` | 主动标记某函数不参与编译（明确边界，避免意外断点） |

```python
torch.compile(model, fullgraph=True)   # 有一个断点就报错，逼你改干净
```

> 不是所有断点都能消除（比如调用外部 C 扩展），这类属于"不可克服"的断点，接受它、把 compile 只用在能捕获的核心计算上即可。

#### 五、guard 与重编译：为什么有时越编译越慢

Dynamo 捕获图时会对"这次看到的条件"做假设，并把假设写成 **guard**（守卫）：输入 shape 是多少、dtype 是什么、某个全局值是多少……运行时每次调用都会检查这些 guard：

- **guard 全通过** → 直接复用已编译的图，很快；
- **有 guard 失败** → 说明输入条件变了，Dynamo **重新编译**一张新图。

重编译有上限（`torch._dynamo.config.recompile_limit`，默认 8 次），超过后不再编译、回退 eager 并告警。

**导致反复重编译的典型场景**：
- 输入 shape 频繁变化（如可变 batch size、变长序列）；
- 循环里某个 Python 每次新建的对象被当作常量捕获。

#### 动态 shape：避免 shape 一变就重编译

如果 shape 变化是常态（CV 的可变图像尺寸、NLP 的可变序列长度），开启动态 shape 让一张图适配多种 shape：

```python
torch.compile(model, dynamic=True)     # 让 Dynamo 生成动态 shape 的图
```

开启后，shape 在约 2 倍以内变化通常不会触发重编译（官方文档：变化小于 factor of 2 时避免重编译）。也可以用 `torch._dynamo.mark_dynamic(tensor, dim)` 精确标记"这个维度是动态的"。

#### 诊断重编译

```bash
TORCH_LOGS=recompiles python train.py      # 打印哪条 guard 失败、触发重编译
```

配合 `torch._dynamo.utils.compile_times()` 看各阶段编译耗时，`TORCH_COMPILE_DEBUG=1` 让 Inductor 输出完整的调试目录。

#### 六、三种 mode：编译策略的取舍

`torch.compile` 的 `mode` 参数决定 Inductor 怎么优化，本质是"编译时间 ↔ 运行性能 ↔ 显存"的权衡：

| mode | 策略 | 代价 | 适合场景 |
|------|------|------|---------|
| `default` | 平衡编译时间与运行性能 | 适中 | 默认首选，大多数情况 |
| `reduce-overhead` | 用 CUDA Graph 减少 kernel 启动开销 | 吃更多显存、要求输入 shape 固定、部分需 CPU-GPU 同步的操作不支持 | 小模型、小 batch，延迟被启动开销主导 |
| `max-autotune` | 编译期对 tile size、warp 数、融合策略做 autotune，选最快 | 编译时间最长 | 追求极致性能、shape 稳定、能忍受长编译 |

```python
torch.compile(model, mode="default")
torch.compile(model, mode="reduce-overhead")   # 小 batch 低延迟
torch.compile(model, mode="max-autotune")      # 编译久，性能上限高
```

还有两个**调试用**后端：`eager`（只测 Dynamo 捕获）、`aot_eager`（测到 AOTAutograd 为止，不生成代码）。当编译出错时，用它们逐级定位问题出在哪一层——是图捕获、求导追踪，还是代码生成。

> 经验：先从 `default` 开始；确认是启动开销主导再试 `reduce-overhead`；最终压榨性能且在固定硬件上长期跑，再上 `max-autotune`。

#### 七、调试工具箱

编译出问题时，靠"猜"很难定位。官方文档提供了一组可观测工具：

| 工具 | 作用 |
|------|------|
| `TORCH_LOGS=graph_breaks` | 打印所有图断裂及原因 |
| `TORCH_LOGS=recompiles` | 打印导致重编译的失败 guard |
| `TORCH_TRACE` + `tlparse` | 记录完整编译过程，事后解析 |
| `torch._dynamo.explain()` | 汇总图数量、断点、算子数、guard |
| `torch._dynamo.utils.compile_times()` | 各阶段编译耗时统计 |
| `TORCH_COMPILE_DEBUG=1` | Inductor 输出调试目录（IR、生成代码、可视化） |
| `fullgraph=True` | 有断点即报错，用于" forcing 定位" |
| `mode="eager" / "aot_eager"` | 逐级跳过后端，定位出错层 |

**典型排错流程**：编译没生效/变慢 → 先用 `explain()` 看断点数 → `TORCH_LOGS=graph_breaks` 找断点原因 → 改写代码或加 `fullgraph=True` 逼近零断点 → 若报编译错误，用 `aot_eager` 后端定位是哪一层的问题。

In [ ]:
import torch
import torch._dynamo as dynamo

# 演示 compile_times：查看各函数捕获阶段的耗时统计
def f(x):
    return (x * 2 + 1).relu()

cf = torch.compile(f)
y = cf(torch.randn(8))          # 触发捕获
y = cf(torch.randn(8))          # 复用
try:
    print(dynamo.utils.compile_times())
except Exception as e:
    print("compile_times 读取失败（版本差异）:", type(e).__name__, e)

# 生产环境：若不想在预热后出现意外重编译，可用已编译图固化
# frozen = dynamo.run(f)  # 只复用已有编译图，不再生成新图

#### 八、与其他技术的配合与边界

#### 8.1 与混合精度 AMP 配合

`torch.compile` 与 `torch.autocast` 可以叠加：AMP 负责用低精度计算，compile 负责融合算子，两者收益可累加。

```python
with torch.autocast("cuda", dtype=torch.bfloat16):
    out = compiled_model(x)
```

#### 8.2 与分布式训练配合

DDP + compile 是常见组合；但也有专门的编译并行方案。注意 compile 通常包装在 DDP **之前**（先编译单个模型，再交给 DDP 同步梯度），具体以版本文档为准。

#### 8.3 与 torch.export 的区别（对照第 07 章）

| 维度 | `torch.compile` | `torch.export` |
|------|----------------|---------------|
| 目标 | **原地加速**训练/推理 | **导出静态图**用于部署 |
| 时机 | 运行时 JIT 编译 | 提前导出一次 |
| 输出 | 优化后的 kernel，仍在 PyTorch 进程内执行 | ExportedProgram，可交给别的运行时/后端 |
| 动态性 | 支持动态 shape、图断裂后回退 eager | 更严格，追求完整图 |

一句话：**compile 让模型在 PyTorch 里跑得更快；export 把模型搬出 PyTorch 去别处跑**。两者解决不同问题。

#### 8.4 已知限制

- **不是二进制等价**：融合与低精度可能带来微小数值差异，极少数敏感训练需留意；
- **首次编译开销大**：服务冷启动、benchmark 都要考虑 warmup；
- **第三方扩展不可见**：PyTorch 之外的 C/NumPy 操作会造成断点（NumPy 需 `torch.compiler.wrap_numpy`）；
- **Windows / 老版本支持有限**：本机环境不支持时，代码会回退或报错，属正常现象。

#### 九、实践建议：把 compile 用在刀刃上

一条实用的决策链：

```
1. 先用 profiler 测 → 确认瓶颈是"碎算子 / kernel 启动开销"
        │
        ├─ 否 → 别上 compile，先解决真正瓶颈（数据、显存、同步）
        │
        └─ 是 ↓
2. torch.compile(model, mode="default") 起步
        ↓
3. 用 explain() / TORCH_LOGS 检查断点与重编译
        ↓
4. 改写代码减少断点；shape 多变就 dynamic=True
        ↓
5. 再用 profiler 对比，验证确实变快
        ↓
6. 还想压榨 → 试 reduce-overhead / max-autotune
```

**什么时候不必上 compile**：模型本来就由少量大 kernel 构成（如大矩阵乘为主）、输入 shape 极度多变、只在 CPU 上跑小模型、或者项目处于快速迭代期（编译调试成本高于收益）。工具是手段不是目的——**测量说了算**。

#### 十、本章总结

| 你学到的 | 一句话要点 |
|---------|-----------|
| 加速本质 | 算子融合：把碎小 kernel 拼成大 kernel，减少启动与显存读写 |
| 编译栈三层 | Dynamo 图捕获 → AOTAutograd 前反向图 → Inductor 生成 Triton/C++ 代码 |
| 图断裂 | 无法静态捕获的操作会断图回退 eager，是加速头号杀手；`explain()` 可诊断 |
| guard 与重编译 | 输入条件变化使 guard 失败就重编译；有上限，动态 shape 用 `dynamic=True` |
| 三种 mode | default 平衡 / reduce-overhead 低延迟吃显存 / max-autotune 编译久性能高 |
| 调试 | `TORCH_LOGS`、`explain()`、`compile_times`、`fullgraph=True`、`aot_eager` 逐级定位 |
| 正确 benchmark | 先 warmup、GPU 加 synchronize、多次取平均 |
| 与 export 区别 | compile 原地加速；export 导出静态图去部署 |
| 使用心法 | 先 profiler 定位瓶颈，再决定上不上 compile，用完再测量验证 |

**最终认知**：`torch.compile` 把一个"能否加速"的经验问题，变成了可观测、可诊断的工程流程——图捕获、guard、断点、mode 都有工具可查。掌握这套流程，你就能在真实项目里判断它到底该不该用、用了为什么快（或为什么不快）。